<div dir="rtl">

# HM_02 — פתרון: מעבדה: לממש ולמדוד את המנגנון של מודל שפה

**הקורס:** מודלי שפה וסוכנים ממוחשבים למסלול מערכות מידע · **מפגש 2:** איך מודל שפה "חושב" קוד?

**שם:** פתרון למרצה — לא להפצה לסטודנטים

---

## מה עושים במעבדה

במפגש 2 למדנו שרשרת של מושגים: **טקסט ← טוקנים ← embeddings ← התפלגות על הטוקן הבא ← בחירה (טמפרטורה)**, ולמה בסוף השרשרת הזו נוצרות **הזיות**. במעבדה הזו תממשו בעצמכם את הפונקציות שבלב השרשרת, ותשתמשו בהן כדי לבדוק במדידה את הטענות מההרצאה.

| חלק | נושא | מה מממשים | זמן |
|---|---|---|---|
| א' | טוקניזציה (BPE) | `count_pairs`, `merge_pair` | 12 דק' |
| ב' | Embeddings | `cosine` | 5 דק' |
| ג' | טמפרטורה | `softmax`, `p_greedy_path` | 10 דק' |
| ד' | הזיות | `audit` | 9 דק' |
| ה' | ניסוי חבילות | `jaccard` | 7 דק' |
| ו' | מסקנה | — | 2 דק' |

## איך עובדים

1. **מממשים** כל פונקציה שמסומנת `# TODO` — **בעצמכם, בלי כלי AI**. המטרה היא להבין את המנגנון, לא לקבל קוד. (בחלק ה' משתמשים בכלי AI, כי הוא נושא הניסוי.)
2. **מריצים את תא הבדיקה** שאחרי כל פונקציה. כשהמימוש נכון יודפס `✓`; אחרת תופיע הודעה שמסבירה מה נכשל.
3. **לפני כל מדידה כותבים תחזית** בתא המיועד לכך. ההבדל בין התחזית לתוצאה הוא חלק מהציון — תחזית "שגויה" עם ניתוח טוב שווה יותר מתחזית "נכונה" בלי הסבר.
4. **עונים על שאלות הניתוח** בתאי ה"תשובה". תשובה טובה מסבירה **במונחי המנגנון** מההרצאה.

## הגשה

בסוף: **Runtime → Restart session and run all**, ודאו שתא הבדיקה המסכם בסוף המחברת מראה `✓` לכל הפונקציות, ואז **File → Download → Download .ipynb** והגישו את הקובץ.

</div>

In [1]:
# תא הכנה — הריצו פעם אחת בתחילת העבודה
import math
import random
import inspect
from collections import Counter

import requests

print("✓ מוכן")

✓ מוכן


<div dir="rtl">

---

# חלק א' — טוקניזציה: לממש BPE בעצמכם (12 דק')

## הרקע

מודל שפה לא קורא אותיות או מילים — הוא קורא **טוקנים**: חתיכות טקסט מתוך "אוצר מילים" קבוע. אוצר המילים נבנה **לפני** אימון המודל, באלגוריתם שנקרא **BPE (Byte Pair Encoding)**:

1. מתחילים כשכל מילה מפורקת לתווים בודדים.
2. סופרים את כל **הצמדים הסמוכים** של טוקנים בכל הקורפוס.
3. **מאחדים** את הצמד השכיח ביותר לטוקן חדש — בכל המקומות שבהם הוא מופיע.
4. חוזרים לשלב 2, עד שמגיעים לגודל אוצר המילים הרצוי.

התוצאה: רצפים **שכיחים** בנתונים הופכים לטוקן אחד, ורצפים **נדירים** נשארים מפורקים לחתיכות קטנות.

## הייצוג

הקורפוס הוא רשימה של זוגות `(tokens, freq)`: מילה מפורקת לרשימת טוקנים, ומספר הפעמים שהמילה מופיעה. למשל, המילה `tax` שמופיעה 3 פעמים, לפני כל מיזוג:

`(["t", "a", "x"], 3)`

## הנוסחה — ספירת צמדים

כל מילה $w$ בקורפוס מיוצגת כרצף טוקנים $w = (w_1, \dots, w_{\lvert w \rvert})$, כאשר $w_k$ הוא הטוקן ה-$k$ במילה, ו-$f(w)$ היא השכיחות שלה. לכל זוג טוקנים $(a, b)$, הספירה שלו היא סכום על כל המילים $w$: השכיחות של המילה, כפול מספר המקומות $k$ שבהם $a$ מופיע מיד לפני $b$:

$$\text{count}(a, b) = \sum_{w} f(w) \cdot \left|\{\, k \;:\; w_k = a,\ w_{k+1} = b \,\}\right|$$

(הקבוצה $\{k : \dots\}$ היא כל המקומות במילה שבהם הזוג מתחיל, ו-$\lvert\cdot\rvert$ סביבה הוא מספר האיברים בה.)

**דוגמה ידנית:** בקורפוס `[(["a", "b", "a", "b"], 2)]` יש בתוך המילה שלושה צמדים סמוכים: `(a,b)`, `(b,a)`, `(a,b)`. כלומר `(a,b)` מופיע פעמיים ו-`(b,a)` פעם אחת. כפול השכיחות 2: `{("a","b"): 4, ("b","a"): 2}`.

## מיזוג צמד

סורקים את רשימת הטוקנים **משמאל לימין**. כשמוצאים את הצמד — מחליפים את שני הטוקנים בטוקן אחד (שרשור של שניהם) ו**מדלגים** על שניהם. טוקן שכבר אוחד לא משתתף שוב באותו מעבר:

- `merge_pair(["a","b","a","b"], ("a","b"))` ← `["ab", "ab"]`
- `merge_pair(["a","a","a"], ("a","a"))` ← `["aa", "a"]` (האות השלישית נשארת לבד)

**כלים שימושיים בפייתון:** `Counter()` הוא מילון שערך ברירת המחדל שלו 0, כך ש-`counts[key] += 1` עובד גם למפתח חדש. `zip(tokens, tokens[1:])` נותן את כל הצמדים הסמוכים ברשימה.

</div>

In [2]:
def count_pairs(corpus):
    """Counts every pair of ADJACENT tokens in the corpus, weighted by word frequency.
    corpus: list of (tokens, freq), e.g. [(["a", "b", "a"], 2)]
    returns: Counter mapping (left, right) -> total count, e.g. {("a","b"): 2, ("b","a"): 2}
    """
    counts = Counter()
    for tokens, freq in corpus:
        for a, b in zip(tokens, tokens[1:]):      # all adjacent pairs (w_k, w_{k+1})
            counts[(a, b)] += freq                # each occurrence weighted by f(w)
    return counts


def merge_pair(tokens, pair):
    """Replaces every occurrence of the adjacent pair (left, right) in tokens with left+right.
    Scans left to right; a token that was just merged is not reused.
    e.g. merge_pair(["a","b","a","b"], ("a","b")) -> ["ab", "ab"]
         merge_pair(["a","a","a"],     ("a","a")) -> ["aa", "a"]
    """
    out, k = [], 0
    while k < len(tokens):
        if k + 1 < len(tokens) and (tokens[k], tokens[k + 1]) == pair:
            out.append(tokens[k] + tokens[k + 1])  # merge, and skip both tokens
            k += 2
        else:
            out.append(tokens[k])
            k += 1
    return out

In [3]:
# בדיקה — חלק א'
def check_a():
    assert count_pairs([(["a", "b", "a", "b"], 2)]) == Counter({("a", "b"): 4, ("b", "a"): 2}), \
        "count_pairs: הספירה על הדוגמה הידנית שגויה"
    assert count_pairs([(["a", "b"], 1), (["a", "b"], 3)]) == Counter({("a", "b"): 4}), \
        "count_pairs: צריך לסכם על פני כל המילים בקורפוס"
    assert merge_pair(["a", "b", "a", "b"], ("a", "b")) == ["ab", "ab"], "merge_pair: מיזוג בסיסי"
    assert merge_pair(["a", "a", "a"], ("a", "a")) == ["aa", "a"], \
        "merge_pair: אחרי מיזוג צריך לדלג על שני הטוקנים"
    assert merge_pair(["x", "y"], ("a", "b")) == ["x", "y"], "merge_pair: צמד שלא קיים — לא משנים כלום"
    assert merge_pair([], ("a", "b")) == [], "merge_pair: רשימה ריקה"
    return True

check_a() and print("✓ חלק א' — המימוש עובר את הבדיקות")

✓ חלק א' — המימוש עובר את הבדיקות


<div dir="rtl">

## אימון מפענח צעצוע

לפניכם קורפוס קטן של שמות משתנים, עם מספר ההופעות של כל אחד. התא הבא מריץ 10 מיזוגים בעזרת הפונקציות שלכם, ואז מפרק כמה מילים חדשות לפי המיזוגים שנלמדו.

**תחזית (לפני ההרצה):** אילו מהמילים יהפכו לטוקן אחד אחרי 10 מיזוגים, ואילו יישארו מפורקות? איך יתפרק `price_with_vat`?

**התחזית שלי (פתרון):** `total` ו-`price` יהפכו לטוקן אחד, כי הם השכיחים בקורפוס; `vat` ו-`with` (שכיחות 1) יישארו מפורקים לתווים. `price_with_vat` יתפרק ל-`price` ואחריו תווים בודדים: `['price', '_', 'w', 'i', 't', 'h', '_', 'v', 'a', 't']`.

</div>

In [4]:
words = {"total": 6, "total_price": 4, "total_tax": 3, "price": 5, "tax": 3, "vat": 1, "with": 1}
corpus = [(list(w), f) for w, f in words.items()]
merges = []
for step in range(10):
    pairs = count_pairs(corpus)
    best = max(pairs, key=lambda p: (pairs[p], p))     # השכיח ביותר (שובר שוויון באופן קבוע)
    merges.append(best)
    corpus = [(merge_pair(t, best), f) for t, f in corpus]
    print(f"מיזוג {step + 1}: {best[0]!r} + {best[1]!r}  (שכיחות {pairs[best]})")


def tokenize(word, merges):
    """Applies the learned merges, in the order they were learned."""
    tokens = list(word)
    for pair in merges:
        tokens = merge_pair(tokens, pair)
    return tokens


print()
for w in ["total_price", "price_with_vat", "total_vat", "ממוצע"]:
    print(f"{w:16} → {tokenize(w, merges)}")

מיזוג 1: 't' + 'a'  (שכיחות 19)
מיזוג 2: 'ta' + 'l'  (שכיחות 13)
מיזוג 3: 't' + 'o'  (שכיחות 13)
מיזוג 4: 'to' + 'tal'  (שכיחות 13)
מיזוג 5: 'r' + 'i'  (שכיחות 9)
מיזוג 6: 'ri' + 'c'  (שכיחות 9)
מיזוג 7: 'ric' + 'e'  (שכיחות 9)
מיזוג 8: 'p' + 'rice'  (שכיחות 9)
מיזוג 9: 'total' + '_'  (שכיחות 7)
מיזוג 10: 'ta' + 'x'  (שכיחות 6)

total_price      → ['total_', 'price']
price_with_vat   → ['price', '_', 'w', 'i', 't', 'h', '_', 'v', 'a', 't']
total_vat        → ['total_', 'v', 'a', 't']
ממוצע            → ['מ', 'מ', 'ו', 'צ', 'ע']


<div dir="rtl">

## השוואה למפענח אמיתי

התא הבא משתמש במפענח `o200k_base` — המפענח של מודלים של OpenAI מ-2024. הוא נבנה באותו עיקרון בדיוק, רק על כמות עצומה של טקסט ועם כמאתיים אלף מיזוגים. הפלט מציג כל טוקן מופרד בקו `|`.

**פלט צפוי (נמדד):** `calculate_total_price_with_vat` — 6 טוקנים: `calculate | _total | _price | _with | _v | at`. `calcTotPrcVat` — 5 טוקנים: `calc | Tot | P | rc | Vat`. המשפט באנגלית — 12 טוקנים (טוקן למילה). המשפט בעברית — 15 טוקנים: `כתוב | פונ | ק | ציה | שמ | חש | בת | ממ | וצ | ע | של | רש | ימת | מספר | ים`.

</div>

In [ ]:
!pip install -q tiktoken
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
for t in ["calculate_total_price_with_vat",
          "calcTotPrcVat",
          "Write a function that computes the average of a list of numbers",
          "כתוב פונקציה שמחשבת ממוצע של רשימת מספרים"]:
    ids = enc.encode(t)
    print(f"{len(ids):3} טוקנים | {' | '.join(enc.decode([i]) for i in ids)}")

<div dir="rtl">

## שאלות ניתוח — חלק א'

**(א)** במפענח הצעצוע שלכם, למה `total` ו-`price` הפכו לטוקן אחד, ו-`vat` נשאר `v | a | t`? מצאו במפענח האמיתי את אותה תופעה בדיוק.

**תשובה:** `total` מופיע 13 פעמים בקורפוס ($6+4+3$, כולל בתוך `total_price` ו-`total_tax`) ו-`price` 9 פעמים ($5+4$). לכן הזוגות שלהם מנצחים בשלב ה-$\arg\max$ ונמזגים. `vat` מופיע פעם אחת בלבד, ואף זוג שלו לא מגיע לראש הספירה ב-10 מיזוגים — הוא נשאר `v | a | t`. במפענח האמיתי: `calculate | _total | _price | _with | _v | at` — `_total` ו-`_price` הם טוקן אחד, ו-`vat` נשבר ל-`_v | at`. אותה תופעה בדיוק.

**(ב)** למה `ממוצע` נשאר מפורק לגמרי במפענח הצעצוע? מה זה מסביר על עברית במפענחים אמיתיים?

**תשובה:** אין בקורפוס הצעצוע אף מילה בעברית מלבד `ממוצע` עצמה (שלא נכללה באימון), ולכן אף זוג של אותיות עבריות לא נספר ולא נמזג — המילה נשארת ברמת התו. מפענחים אמיתיים נבנים על קורפוס שרובו אנגלית; רצפים עבריים נדירים יותר, ולכן פחות מהם נמזגים — וזה המנגנון שמאחורי $\rho_\text{he} \approx 2$ מול $\rho_\text{en} \approx 1$.

**(ג) קיבולת.** לפי הפלט של המפענח האמיתי, חשבו טוקנים-למילה בעברית ובאנגלית (ספרו את המילים בכל משפט). **חלון הקשר** הוא מספר הטוקנים המרבי שהמודל מעבד בבת אחת — והוא כולל גם את התשובה. בחלון של 128,000 טוקנים, שמתוכם 8,000 שמורים לתשובה: כמה מילים בעברית נכנסות, וכמה באנגלית? (אפשר לחשב בתא קוד.)

**תשובה:** $\rho_\text{en} = 12/12 = 1.0$, $\rho_\text{he} = 15/7 \approx 2.14$. עם $L_\text{free} = 128{,}000 - 8{,}000 = 120{,}000$: כ-120,000 מילים באנגלית, וכ-56,000 מילים בעברית (ראו תא החישוב).

**(ד)** `calcTotPrcVat` קצר פי 2.3 בתווים מ-`calculate_total_price_with_vat`. כמה טוקנים הוא חוסך? האם שווה לקצר שמות משתנים "כדי לחסוך טוקנים"?

**תשובה:** הקיצור חוסך טוקן אחד בלבד (6 ← 5). המחיר: `Tot`, `P`, `rc` הם רצפים בלי משמעות ברורה, ולכן ה-embeddings שלהם נותנים למודל (ולקורא) הקשר חלש בהרבה משל `_total` ו-`_price`. לא שווה: החיסכון זניח, ואיכות ההקשר יורדת.

</div>

In [6]:
# תא חישוב לשאלה (ג) — חלק א'
rho_en = 12 / 12          # T1 באנגלית: 12 טוקנים, 12 מילים
rho_he = 15 / 7           # T1 בעברית: 15 טוקנים, 7 מילים
L_free = 128_000 - 8_000  # L - n_answer
print(f"rho_en = {rho_en:.2f}, rho_he = {rho_he:.2f}")
print(f"מילים באנגלית: {L_free / rho_en:,.0f} | מילים בעברית: {L_free / rho_he:,.0f}")

rho_en = 1.00, rho_he = 2.14
מילים באנגלית: 120,000 | מילים בעברית: 56,000


<div dir="rtl">

---

# חלק ב' — Embeddings: לממש דמיון קוסינוס (5 דק')

## הרקע

אחרי הטוקניזציה, כל טוקן ממופה ל**וקטור** — רשימה ארוכה של מספרים, שאפשר לחשוב עליה כ"כתובת" של נקודה במרחב רב-ממדי. המיקומים נלמדים באימון, כך שטוקנים שמופיעים **בהקשרים דומים** מקבלים מיקומים **קרובים**. קרבה במרחב ≈ קרבה במשמעות.

## הנוסחה

מודדים קרבה בעזרת **דמיון קוסינוס** — הקוסינוס של הזווית בין שני הווקטורים:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert}, \qquad \mathbf{a} \cdot \mathbf{b} = \sum_{k=1}^{d} a_k b_k, \qquad \lVert \mathbf{a} \rVert = \sqrt{\mathbf{a} \cdot \mathbf{a}} = \sqrt{\sum_{k=1}^{d} a_k^2}$$

כאן $\mathbf{a} = (a_1, \dots, a_d)$ ו-$\mathbf{b} = (b_1, \dots, b_d)$ הם שני וקטורים באורך $d$ (למשל שני embeddings), ו-$a_k$ הוא הרכיב ה-$k$ של $\mathbf{a}$.

- $\mathbf{a} \cdot \mathbf{b}$ — **מכפלה סקלרית**: מכפילים איבר-איבר ומסכמים.
- $\lVert \mathbf{a} \rVert$ — **אורך** (נורמה) של הווקטור.
- התוצאה בין $-1$ ל-$1$: ערך **1** = אותו כיוון בדיוק; **0** = ניצבים (אין קשר); **‎−1** = כיוונים הפוכים.

**דוגמה ידנית:** $\mathbf{a} = (1, 0)$, $\mathbf{b} = (1, 1)$: $\mathbf{a} \cdot \mathbf{b} = 1 \cdot 1 + 0 \cdot 1 = 1$, $\lVert \mathbf{a} \rVert = 1$, $\lVert \mathbf{b} \rVert = \sqrt{2}$, ולכן $\cos = 1/\sqrt{2} \approx 0.707$ (זווית של 45°).

</div>

In [7]:
def cosine(a, b):
    """Cosine similarity of two equal-length vectors a and b:
    (a · b) / (|a| * |b|), where a · b = sum of a[k]*b[k] and |a| = sqrt(a · a).
    """
    dot = sum(a_k * b_k for a_k, b_k in zip(a, b))           # a · b
    norm_a = math.sqrt(sum(a_k * a_k for a_k in a))           # |a|
    norm_b = math.sqrt(sum(b_k * b_k for b_k in b))           # |b|
    return dot / (norm_a * norm_b)

In [8]:
# בדיקה — חלק ב' (וקטורי הצעצוע מההרצאה: [חיה?, מחמד?, כלי תחבורה?])
def check_b():
    dog, cat, car = [0.90, 0.80, 0.10], [0.85, 0.75, 0.20], [0.10, 0.20, 0.95]
    assert abs(cosine([1, 0], [1, 1]) - 1 / math.sqrt(2)) < 1e-12, "cosine: הדוגמה הידנית (0.707)"
    assert round(cosine(dog, cat), 3) == 0.996, "cosine(כלב, חתול) צריך להיות 0.996"
    assert round(cosine(dog, car), 3) == 0.293, "cosine(כלב, מכונית) צריך להיות 0.293"
    assert abs(cosine([1, 0], [0, 1])) < 1e-12, "וקטורים ניצבים — דמיון 0"
    assert abs(cosine([2, 2], [1, 1]) - 1) < 1e-12, "אותו כיוון, אורך שונה — דמיון 1"
    return True

check_b() and print("✓ חלק ב' — המימוש עובר את הבדיקות")

✓ חלק ב' — המימוש עובר את הבדיקות


<div dir="rtl">

## שאלות ניתוח — חלק ב'

**(א) תחזית ואז חישוב:** מה יחזיר `cosine([1, 2, 3], [-1, -2, -3])`? כתבו תחזית, הריצו בתא שלמטה, והסבירו את משמעות הערך.

**התחזית שלי (פתרון):** $-1$.

**תשובה:** `-1.0`. שני הווקטורים מצביעים לכיוונים הפוכים בדיוק: $\mathbf{b} = -\mathbf{a}$, ולכן $\mathbf{a} \cdot \mathbf{b} = -\lVert \mathbf{a} \rVert^2$ ו-$\cos = -1$.

**(ב)** הבדיקה האחרונה אומרת ש-`[2, 2]` ו-`[1, 1]` "זהים" (דמיון 1), למרות שהמרחק ביניהם אינו 0. למה זו תכונה **רצויה** כשמשווים משמעות?

**תשובה:** כי משמעות מיוצגת על ידי **הכיוון** של הווקטור, ואילו האורך יכול להיות מושפע מגורמים אחרים (למשל שכיחות הטוקן). קוסינוס מתעלם מהאורך ($\cos(\lambda\mathbf{a}, \mathbf{b}) = \cos(\mathbf{a}, \mathbf{b})$ לכל $\lambda > 0$), ולכן משווה רק "על מה הווקטור מצביע".

**(ג) הקשר להזיות:** נניח שהדמיון בין ה-embedding של `get` לזה של `get_value` גבוה מאוד. איך זה יכול להסביר את ההזיה שתמצאו בחלק ד'?

**תשובה:** אם $\cos(\mathbf{e}_\texttt{get}, \mathbf{e}_\texttt{get\_value})$ גבוה, שני השמות "נראים" למודל כמעט אותו דבר. כשהמודל בוחר את ההמשך אחרי `data.`, גם `get_value` מקבל ציון גבוה — וייתכן שייבחר, למרות שלמילון ב-Python אין מתודה כזו. זו חוליה 3 בשרשרת ההזיות (קרבה במרחב).

</div>

In [9]:
# תא לשאלה (א)
print(cosine([1, 2, 3], [-1, -2, -3]))

-1.0


<div dir="rtl">

---

# חלק ג' — טמפרטורה: לממש softmax, ולמדוד מה קורה לתשובה שלמה (10 דק')

## הרקע

בכל צעד, המודל נותן **ציון גולמי** (logit) $z_t$ לכל טוקן אפשרי $t$. הציונים הופכים להסתברויות, ונבחר טוקן אחד **בדגימה** מתוך ההתפלגות. הטוקן שנבחר מתווסף לקלט, וחוזרים על התהליך — טוקן אחר טוקן.

## הנוסחה — softmax עם טמפרטורה

$$p(t) = \frac{e^{z_t / T}}{\sum_{t'} e^{z_{t'} / T}}$$

$p(t)$ היא ההסתברות שייבחר הטוקן $t$, ו-$t'$ הוא אינדקס סכימה שרץ על כל הטוקנים המועמדים (כך שהמכנה זהה לכל $t$, וסכום ההסתברויות הוא 1).

- $T$ קטן (למשל 0.2) ← ההבדלים בין הציונים מתעצמים ← כמעט תמיד נבחר הטוקן המוביל.
- $T = 1$ ← ההתפלגות כפי שהמודל חישב.
- $T$ גדול (למשל 1.5) ← ההתפלגות "שטוחה" ← אפשרויות פחות סבירות נבחרות יותר.

## בעיה נומרית — גלישה (overflow)

מספר עשרוני (`float`) במחשב לא יכול להיות גדול מכ-$1.8 \times 10^{308}$. כבר $e^{710}$ גדול מזה, ולכן `math.exp(1000)` זורק `OverflowError`. ציונים כאלה לא נדירים במודלים אמיתיים, ולכן מימוש "ישיר" של הנוסחה **שבור**.

**הרמז לפתרון:** אם מחסירים **אותו קבוע** $c$ מכל הציונים, התוצאה של softmax **לא משתנה** (תוכיחו זאת בשאלה ב'). בחרו $c$ שמבטיח שאף מעריך לא יהיה גדול מ-0. (מה הערך של $e^0$? ומה המעריך הגדול ביותר שיכול להופיע?)

## המודל לתשובה שלמה

תשובה של 200 טוקנים מורכבת מ-190 טוקנים **קלים** (כמעט כפויים — למשל סוגר אחרי פתיחת סוגריים) ו-10 **נקודות החלטה** (כמו בחירת ההמשך של `for i in ...`). נשאל: מה ההסתברות שכל 200 הטוקנים יהיו בדיוק הבחירה המובילה — כלומר שהתשובה תהיה **זהה** לתשובה שמתקבלת ב-$T = 0$?

נסמן ב-$\pi_\text{easy}$ את ההסתברות של הטוקן המוביל בטוקן קל, וב-$\pi_\text{dec}$ את ההסתברות של הטוקן המוביל בנקודת החלטה (שתיהן מחושבות ב-softmax, עם אותה $T$). אם הטוקנים **בלתי-תלויים**, ההסתברות שכולם יהיו המובילים היא מכפלה:

$$P(\text{תשובה זהה}) = \pi_\text{easy}^{\,n_\text{easy}} \cdot \pi_\text{dec}^{\,n_\text{dec}}$$

**הערה:** בשתי רשימות הציונים שנשתמש בהן, הציון הראשון הוא הגבוה ביותר — כך שההסתברות של הטוקן המוביל היא פשוט האיבר הראשון ברשימת ההסתברויות.

</div>

In [10]:
def softmax(z, T):
    """Turns raw scores z into probabilities with temperature T:
    p(t) = exp(z_t / T) / (sum over all candidates t' of exp(z_t' / T))
    Must not overflow for large scores (e.g. z = [1000, 999]).
    Hint: subtracting the same constant c from every z_t does not change the result.
    """
    c = max(z)                                   # with c = max, every exponent is <= 0
    e = [math.exp((z_t - c) / T) for z_t in z]
    total = sum(e)
    return [e_t / total for e_t in e]


def p_greedy_path(T, z_easy, z_decision, n_easy=190, n_decision=10):
    """Probability that EVERY token of an answer is the top choice, assuming independent tokens:
    n_easy "easy" tokens with scores z_easy and n_decision "decision" tokens with scores z_decision.
    The top choice is the first score in each list.
    """
    pi_easy = softmax(z_easy, T)[0]
    pi_dec = softmax(z_decision, T)[0]
    return pi_easy ** n_easy * pi_dec ** n_decision

In [11]:
# בדיקה — חלק ג'
z_decision = [4.0, 2.7, 1.7, 1.2, 0.5]   # נקודת החלטה — הטבלה מההרצאה ("for i in ...")
z_easy = [8.0, 2.0, 1.0]                  # טוקן "קל", כמעט כפוי

def check_c():
    p = softmax(z_decision, 1.0)
    assert abs(sum(p) - 1) < 1e-9, "softmax: סכום ההסתברויות חייב להיות 1"
    assert round(p[0], 3) == 0.683, "softmax ב-T=1: האפשרות המובילה צריכה לקבל 0.683"
    assert round(softmax(z_decision, 0.2)[0], 3) == 0.998, "softmax ב-T=0.2: המובילה צריכה לקבל 0.998"
    assert softmax([1000, 999], 1.0)[0] > 0.7, "softmax: לא יציב נומרית (ראו את הרמז)"
    assert round(p_greedy_path(1.0, z_easy, z_decision), 4) == 0.0116, "p_greedy_path ב-T=1 צריך להיות 0.0116"
    return True

check_c() and print("✓ חלק ג' — המימוש עובר את הבדיקות")

✓ חלק ג' — המימוש עובר את הבדיקות


<div dir="rtl">

## מדידה

**תחזית (לפני ההרצה):** ב-$T = 1$, מה הסיכוי (באחוזים) שתשובה של 200 טוקנים תהיה זהה לתשובה המובילה?

**התחזית שלי (פתרון):** כ-1% (החישוב: $0.0116$).

התא הבא עושה שני דברים: (1) דוגם 1,000 פעמים טוקן בודד ומשווה את השכיחות שנמדדה להסתברות התאורטית; (2) מסמלץ 1,000 "תשובות" שלמות ובודק בכמה מהן כל הטוקנים היו המובילים — ומשווה לחישוב האנליטי שלכם.

</div>

In [12]:
random.seed(0)
for T in [0.2, 0.7, 1.0, 1.5]:
    p = softmax(z_decision, T)
    sample = random.choices(range(len(p)), weights=p, k=1000)
    print(f"T={T}: P(מוביל) תאורטי={p[0]:.3f}  נמדד ב-1000 דגימות={sample.count(0)/1000:.3f}")


def simulate(T, runs=1000, n_easy=190, n_decision=10):
    pe, pd = softmax(z_easy, T), softmax(z_decision, T)
    same = 0
    for _ in range(runs):
        same += all(random.random() < pe[0] for _ in range(n_easy)) and \
                all(random.random() < pd[0] for _ in range(n_decision))
    return same / runs


print()
random.seed(1)
for T in [0.2, 0.7, 1.0]:
    print(f"T={T}: אנליטי={p_greedy_path(T, z_easy, z_decision):.4f}  סימולציה={simulate(T):.3f}")

T=0.2: P(מוביל) תאורטי=0.998  נמדד ב-1000 דגימות=0.998
T=0.7: P(מוביל) תאורטי=0.821  נמדד ב-1000 דגימות=0.804
T=1.0: P(מוביל) תאורטי=0.683  נמדד ב-1000 דגימות=0.679
T=1.5: P(מוביל) תאורטי=0.530  נמדד ב-1000 דגימות=0.510

T=0.2: אנליטי=0.9850  סימולציה=0.984
T=0.7: אנליטי=0.1325  סימולציה=0.116
T=1.0: אנליטי=0.0116  סימולציה=0.016


<div dir="rtl">

## שאלות ניתוח — חלק ג'

**(א)** הנוסחה של `p_greedy_path` מניחה שהטוקנים **בלתי-תלויים**. האם ההנחה סבירה במודל שמייצר טוקן אחר טוקן, כשכל טוקן שנבחר הופך לחלק מהקלט של הבא? מה קורה בפועל אחרי שטוקן אחד "סוטה"?

**תשובה:** לא לגמרי. במודל אוטו-רגרסיבי, כל טוקן שנבחר הופך לחלק מ-$t_{<i}$ של כל הצעדים הבאים. אחרי סטייה אחת ההקשר שונה, ולכן גם ההתפלגויות $p^{(i)}(t)$ בצעדים הבאים שונות — התשובה עלולה "לזלוג" לכיוון אחר לגמרי. ההנחה נותנת קירוב שימושי לסיכוי לתשובה זהה, אבל לא מתארת מה קורה אחרי הסטייה.

**(ב)** הוכיחו את הרמז: כתבו את השורה האלגברית שמראה שחיסור קבוע $c$ מכל הציונים $z_t$ לא משנה את $p(t)$. איזה $c$ בחרתם, ולמה הוא מונע גלישה?

**תשובה:** $\dfrac{e^{(z_t - c)/T}}{\sum_{t'} e^{(z_{t'} - c)/T}} = \dfrac{e^{-c/T} e^{z_t/T}}{e^{-c/T} \sum_{t'} e^{z_{t'}/T}} = \dfrac{e^{z_t/T}}{\sum_{t'} e^{z_{t'}/T}}$ — הגורם $e^{-c/T}$ מצטמצם. בחרנו $c = \max_{t'} z_{t'}$: כל מעריך הוא $\le 0$, ולכן כל איבר הוא $\le e^0 = 1$ ואין גלישה. בלי זה, `math.exp(1000)` זורק `OverflowError: math range error`.

**(ג)** ב-$T = 1$, טוקן קל בוחר את המוביל ב-99.7% מהמקרים. למה התשובה כולה כמעט אף פעם לא זהה לתשובה המובילה? חשבו **בנפרד** את התרומה של 190 הטוקנים הקלים ($\pi_\text{easy}^{190}$) ושל 10 נקודות ההחלטה ($\pi_\text{dec}^{10}$). מי תורם יותר לסטייה?

**תשובה:** $\pi_\text{easy}^{190} \approx 0.53$ ו-$\pi_\text{dec}^{10} \approx 0.022$ (ראו תא החישוב). נקודות ההחלטה תורמות את רוב הסטייה (מכפילות פי 0.022), אבל גם הטוקנים הקלים לבדם מורידים את הסיכוי לכחצי — כי הסתברות קרובה ל-1 בחזקת 190 כבר אינה קרובה ל-1.

**(ד)** הרצתם פרומפט פעמיים וקיבלתם קוד שונה — האם אחת התשובות שגויה? ואם קיבלתם קוד **זהה** פעמיים — מה זה מרמז על ההתפלגות של המודל בשאלה הזו?

**תשובה:** תשובות שונות אינן בהכרח שגויות — ייתכן ששתיהן נכונות, ופשוט נדגמו מסלולים שונים. תשובה **זהה** פעמיים ב-$T = 1$ מרמזת שההתפלגות חדה מאוד בשאלה הזו: מעט נקודות החלטה אמיתיות, ו-$\pi^{(i)}$ קרוב ל-1 כמעט בכל צעד.

</div>

In [13]:
# תא חישוב לשאלה (ג) — חלק ג'
pi_easy = softmax(z_easy, 1.0)[0]
pi_dec = softmax(z_decision, 1.0)[0]
print(f"pi_easy = {pi_easy:.4f} → pi_easy^190 = {pi_easy ** 190:.3f}")
print(f"pi_dec  = {pi_dec:.4f} → pi_dec^10  = {pi_dec ** 10:.4f}")
print(f"מכפלה = {pi_easy ** 190 * pi_dec ** 10:.4f}")

pi_easy = 0.9966 → pi_easy^190 = 0.526
pi_dec  = 0.6832 → pi_dec^10  = 0.0221
מכפלה = 0.0116


<div dir="rtl">

---

# חלק ד' — הזיות: לממש בדיקת קיום, ולגלות מה היא מפספסת (9 דק')

## הרקע

**הזיה** היא פלט שנשמע סביר אבל שגוי. בהרצאה ראינו חמישה סוגי הזיות בקוד:

| סוג | מה קורה | איך נראית השגיאה בדרך כלל |
|---|---|---|
| פונקציה/מתודה | שם שלא קיים | `AttributeError` |
| פרמטר | פרמטר שהפונקציה לא מקבלת | `TypeError` — או **שום שגיאה** |
| חבילה | ספרייה שלא קיימת | `ModuleNotFoundError` / שגיאת התקנה |
| גרסה | קיים, אבל בגרסה אחרת | תלוי |
| התנהגות | הקוד רץ, אבל לא עושה מה שנאמר | **אין שגיאה** |

## הקוד החשוד

הקוד הבא אמור להוריד מידע על הספרייה `requests` מאתר PyPI ולהדפיס אותו. יש בו **ארבע** הזיות. **אל תריצו אותו עדיין.**

<pre dir="ltr">
import requests

URL = "https://pypi.org/pypi/requests/json"
r = requests.fetch(URL)
data = r.to_json()
print(data.get_value("info"))
r2 = requests.get(URL, retries=3)
</pre>

**תחזית (לפני הכול):** לכל שורה חשודה — סוג ההזיה לפי הטבלה, ואיזו חריגה תיזרק.

**התחזית שלי (פתרון):** `requests.fetch` — הזיית פונקציה, `AttributeError`. `r.to_json()` — הזיית מתודה, `AttributeError`. `data.get_value(...)` — הזיית מתודה על מילון, `AttributeError`. `retries=3` — הזיית פרמטר, `TypeError` (`requests` מעבירה את הפרמטר הלאה לפונקציה שבודקת אותו).

## בדיקה סטטית — בלי להריץ את הקוד החשוד

הפונקציה המובנית `hasattr(obj, name)` מחזירה `True` אם לאובייקט `obj` יש תכונה (פונקציה, מתודה, משתנה) בשם `name`, ו-`False` אחרת — **בלי** להפעיל אותה. עליה תבנו כלי בדיקה.

</div>

In [14]:
def audit(obj, names):
    """For each name, reports whether obj really has an attribute with that name.
    returns: dict name -> bool, e.g. audit(dict, ["get", "get_value"]) -> {"get": True, "get_value": False}
    """
    return {name: hasattr(obj, name) for name in names}

In [15]:
# בדיקה — חלק ד', ואז הבדיקה הסטטית של הקוד החשוד
def check_d():
    assert audit(dict, ["get", "get_value"]) == {"get": True, "get_value": False}, "audit: הדוגמה מה-docstring"
    assert audit(list, []) == {}, "audit: רשימת שמות ריקה"
    return True

check_d() and print("✓ חלק ד' — המימוש עובר את הבדיקות\n")
print("requests:         ", audit(requests, ["get", "fetch"]))
print("requests.Response:", audit(requests.Response, ["json", "to_json"]))
print("dict:             ", audit(dict, ["get", "get_value"]))
print("החתימה של requests.get:", inspect.signature(requests.get))

✓ חלק ד' — המימוש עובר את הבדיקות

requests:          {'get': True, 'fetch': False}
requests.Response: {'json': True, 'to_json': False}
dict:              {'get': True, 'get_value': False}
החתימה של requests.get: (url: '_t.UriType', params: '_t.ParamsType' = None, **kwargs: 'Unpack[_t.GetKwargs]') -> 'Response'


<div dir="rtl">

## בדיקה בהרצה

התא הבא מריץ כל שורה חשודה **בנפרד**, בתוך `try/except`, ומדפיס את סוג החריגה ואת ההודעה. (`exec` מריץ מחרוזת כקוד פייתון.)

</div>

In [16]:
URL = "https://pypi.org/pypi/requests/json"
for line in ['r = requests.fetch(URL)',
             'r = requests.get(URL, timeout=10)',
             'data = r.to_json()',
             'data = r.json()',
             'print(data.get_value("info"))',
             'r2 = requests.get(URL, retries=3)']:
    try:
        exec(line)
        print("OK  ", line)
    except Exception as e:
        print("ERR ", line, "->", f"{type(e).__name__}: {e}")

ERR  r = requests.fetch(URL) -> AttributeError: module 'requests' has no attribute 'fetch'
OK   r = requests.get(URL, timeout=10)
ERR  data = r.to_json() -> AttributeError: 'Response' object has no attribute 'to_json'
OK   data = r.json()
ERR  print(data.get_value("info")) -> AttributeError: 'dict' object has no attribute 'get_value'
ERR  r2 = requests.get(URL, retries=3) -> TypeError: Session.request() got an unexpected keyword argument 'retries'


<div dir="rtl">

## תיקון

כתבו גרסה מתוקנת של הקוד החשוד, שמדפיסה את הגרסה האחרונה של `requests` (הנתון נמצא ב-`data["info"]["version"]`). בלי ניסיונות חוזרים — רק הורדה, פענוח JSON והדפסה.

</div>

In [17]:
# הקוד המתוקן
URL = "https://pypi.org/pypi/requests/json"
r = requests.get(URL, timeout=10)     # requests.fetch -> requests.get
data = r.json()                       # r.to_json() -> r.json()
print(data["info"]["version"])        # data.get_value("info") -> data["info"]
                                      # retries=3 removed: requests.get has no such parameter

2.34.2


<div dir="rtl">

## שאלות ניתוח — חלק ד'

**(א)** איזו הזיה ה-`audit` שלכם **לא** יכול לתפוס, ולמה? מה בחתימה של `requests.get` "מסתיר" אותה? (`**kwargs` בחתימה פירושו: "הפונקציה מקבלת כל פרמטר בעל שם, ומעבירה אותו הלאה".)

**תשובה:** `retries`. `audit` בודק שמות של **תכונות** (`hasattr`), לא שמות של **פרמטרים**. והחתימה של `requests.get` מכילה `**kwargs` — כלומר היא מקבלת "כל פרמטר בעל שם" — כך שגם בדיקת החתימה לא פוסלת אותו. רק הרצה או התיעוד תופסים אותו. (בגרסאות חדשות של `requests` החתימה מוצגת כ-`**kwargs: Unpack[GetKwargs]`: כלי בדיקת טיפוסים סטטי כמו mypy יכול לתפוס את `retries`, אבל Python בזמן ריצה רואה רק `**kwargs`.)

**(ב)** `retries` זרק `TypeError` רק כי `requests` מעבירה את `**kwargs` הלאה לפונקציה שבודקת אותם. בתא שלמטה, כתבו פונקציה קצרה משלכם בשם `download(url, **kwargs)` ש**מתעלמת** מפרמטרים לא מוכרים, והראו שהקריאה `download(URL, retries=3)` "עובדת". לאיזה סוג הזיה הפכה ההזיה, ולמה היא מסוכנת יותר?

**תשובה:** ראו התא שלמטה: `download(URL, retries=3)` רץ בלי שום שגיאה — אבל בלי אף ניסיון חוזר. הזיית הפרמטר הפכה ל**הזיית התנהגות**: הקוד "עובד", ההסבר של המודל ("יש 3 ניסיונות חוזרים") שגוי, ושום דבר לא נכשל. לכן היא המסוכנת ביותר — אין שום סימן שמשהו לא בסדר.

**(ג)** השוו לתחזית שכתבתם בתחילת החלק. מה היה שגוי, ומה למדתם?

**תשובה (דוגמה):** תשובה אישית. טעויות נפוצות בתחזית: לצפות ל-`ImportError` על `fetch` (אבל הייבוא של `requests` מצליח — החסרה היא תכונה בתוך המודול, ולכן `AttributeError`), או לצפות ש-`retries` "יעבוד" (בפועל `TypeError`). הלקח: סוג החריגה מלמד **איזו** חוליה נכשלה.

</div>

In [18]:
# הפונקציה download לשאלה (ב)
def download(url, **kwargs):
    """Accepts ANY keyword argument and silently ignores the ones it does not use."""
    return requests.get(url, timeout=10)

r = download(URL, retries=3)          # no error at all
print(r.status_code, "— אין שגיאה, אבל גם אין שום ניסיון חוזר: הזיית פרמטר הפכה להזיית התנהגות")

200 — אין שגיאה, אבל גם אין שום ניסיון חוזר: הזיית פרמטר הפכה להזיית התנהגות


<div dir="rtl">

---

# חלק ה' — ניסוי: המלצות חבילות של מודל אמיתי (7 דק')

## הרקע

במחקר שהוצג בהרצאה, מודלים מסחריים המליצו בממוצע על לפחות 5.2% חבילות **שלא קיימות**, והרבה מהשמות הבדויים **חזרו** בהרצות חוזרות. תוקף יכול לרשום שם כזה כחבילה אמיתית עם קוד זדוני ולחכות שמישהו יתקין אותה (**slopsquatting**). בניסוי הזה תבדקו כלי AI אמיתי.

## מדד העקביות — דמיון ג'קארד

כדי למדוד עד כמה שתי רשימות המלצות דומות, מתייחסים אליהן כאל **קבוצות** (בלי חשיבות לסדר ולכפילויות) ומחשבים:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

- $|A \cap B|$ — כמה איברים משותפים לשתי הקבוצות (**חיתוך**).
- $|A \cup B|$ — כמה איברים יש בסך הכול באחת מהן לפחות (**איחוד**).
- $J = 1$: קבוצות זהות. $J = 0$: אין אף איבר משותף. **מוסכמה:** שתי קבוצות ריקות — $J = 1$.

**דוגמה ידנית:** $A = \{\text{pandas}, \text{openpyxl}\}$, $B = \{\text{pandas}, \text{xlsxwriter}\}$: חיתוך = 1, איחוד = 3, ולכן $J = 1/3$.

**בפייתון:** `set(a)` הופך רשימה לקבוצה; `a & b` הוא חיתוך, `a | b` הוא איחוד, ו-`len` נותן את הגודל.

</div>

In [19]:
def jaccard(a, b):
    """Jaccard similarity of two collections, treated as sets: |A ∩ B| / |A ∪ B|.
    1.0 = identical sets, 0.0 = nothing in common. Two empty collections -> 1.0.
    """
    A, B = set(a), set(b)
    if not A and not B:
        return 1.0
    return len(A & B) / len(A | B)

In [20]:
# בדיקה — חלק ה'
def check_e():
    assert jaccard(["pandas", "openpyxl"], ["pandas", "xlsxwriter"]) == 1 / 3, "jaccard: הדוגמה הידנית"
    assert jaccard([], []) == 1.0, "jaccard: שתי קבוצות ריקות — 1.0"
    assert jaccard(["a"], ["b"]) == 0.0, "jaccard: אין חיתוך — 0.0"
    assert jaccard(["a", "a", "b"], ["b", "a"]) == 1.0, "jaccard: כפילויות וסדר לא משנים"
    return True

check_e() and print("✓ חלק ה' — המימוש עובר את הבדיקות")

✓ חלק ה' — המימוש עובר את הבדיקות


<div dir="rtl">

## כלי האימות (נתון)

הפונקציה הבאה **בודקת** חבילה ב-PyPI **בלי להתקין אותה**. היא מציגה גם את **תאריך ההעלאה הראשונה** של החבילה. **כלל:** אף פעם לא מתקינים חבילה רק כי מודל המליץ עליה — קודם בודקים.

</div>

In [21]:
def check_pypi(name):
    """Looks a package up on PyPI WITHOUT installing it."""
    r = requests.get(f"https://pypi.org/pypi/{name}/json", timeout=10)
    if r.status_code == 404:
        print(f"{name:25} ✗ לא קיים ב-PyPI")
        return False
    info = r.json()
    uploads = [f["upload_time_iso_8601"][:10] for files in info["releases"].values() for f in files]
    first = min(uploads) if uploads else "?"
    print(f"{name:25} ✓ קיים | גרסה אחרונה {info['info']['version']:10} | הועלה לראשונה {first}")
    return True


for name in ["pandas", "openpyxl", "excel-fast-writer-pro"]:   # בדיקת שפיות
    check_pypi(name)

pandas                    ✓ קיים | גרסה אחרונה 3.0.6      | הועלה לראשונה 2009-12-25


openpyxl                  ✓ קיים | גרסה אחרונה 3.1.5      | הועלה לראשונה 2010-08-11
excel-fast-writer-pro     ✗ לא קיים ב-PyPI


<div dir="rtl">

## הניסוי

הריצו בכלי AI חינמי את הפרומפט הבא **שלוש פעמים, כל פעם בשיחה חדשה** (כדי שההרצות יהיו בלתי-תלויות):

> כתוב קוד Python שקורא קובץ CSV, מסנן שורות שבהן העמודה price גדולה מ-100, ושומר את התוצאה לקובץ Excel. ציין אילו ספריות צריך להתקין.

לכל הרצה, רשמו בתא הבא את רשימת הספריות שהומלצו (בשמות שבהם מתקינים אותן, למשל `openpyxl`), והריצו אותו. **כלי ה-AI ששימש:** _(בפתרון — רשימות להמחשה בלבד; התוצאות האמיתיות משתנות בין סטודנטים וכלים)_

</div>

In [22]:
# בפתרון: רשימות להמחשה בלבד
run1 = ["pandas", "openpyxl"]                              # דוגמה בלבד — יש להחליף בתוצאות אמיתיות
run2 = ["pandas", "xlsxwriter"]
run3 = ["pandas", "openpyxl", "excel-fast-writer-pro"]

runs = [run1, run2, run3]
exists = {}                                   # name -> True/False (each name is checked once)
for i, run in enumerate(runs, 1):
    print(f"--- הרצה {i} ---")
    for name in run:
        if name not in exists:
            exists[name] = check_pypi(name)
        else:
            print(f"{name:25} (נבדק כבר: {'✓ קיים' if exists[name] else '✗ לא קיים'})")

if exists:
    missing = sum(not ok for ok in exists.values())
    print(f"\nחבילות ייחודיות: {len(exists)}, מתוכן לא קיימות: {missing} ({missing / len(exists):.0%})")
    print(f"ג'קארד 1-2: {jaccard(run1, run2):.2f} | 1-3: {jaccard(run1, run3):.2f} | 2-3: {jaccard(run2, run3):.2f}")

--- הרצה 1 ---


pandas                    ✓ קיים | גרסה אחרונה 3.0.6      | הועלה לראשונה 2009-12-25


openpyxl                  ✓ קיים | גרסה אחרונה 3.1.5      | הועלה לראשונה 2010-08-11
--- הרצה 2 ---
pandas                    (נבדק כבר: ✓ קיים)
xlsxwriter                ✓ קיים | גרסה אחרונה 3.2.9      | הועלה לראשונה 2013-02-17
--- הרצה 3 ---
pandas                    (נבדק כבר: ✓ קיים)
openpyxl                  (נבדק כבר: ✓ קיים)
excel-fast-writer-pro     ✗ לא קיים ב-PyPI

חבילות ייחודיות: 4, מתוכן לא קיימות: 1 (25%)
ג'קארד 1-2: 0.33 | 1-3: 0.67 | 2-3: 0.25


<div dir="rtl">

## שאלות ניתוח — חלק ה'

**(א)** קשרו את ערכי ג'קארד שקיבלתם לחלק ג': מה ג'קארד נמוך (או גבוה) מלמד על ההתפלגות של המודל בשאלה הזו?

**תשובה:** בדוגמה: $J_{12} = 0.33$, $J_{13} = 0.67$, $J_{23} = 0.25$ — עקביות נמוכה. כמו בחלק ג': ההמלצה על ספרייה היא "נקודת החלטה" שבה ההתפלגות שטוחה (כמה ספריות סבירות), ולכן הרצות שונות דוגמות המלצות שונות. ג'קארד גבוה (קרוב ל-1) היה מרמז על התפלגות חדה. ג'קארד נמוך = מקום להגביר בדיקה.

**(ב)** האם אפשר להשוות את שיעור החבילות הלא-קיימות שמדדתם לנתון מההרצאה (5.2%)? מה הייתם צריכים כדי להשוות ברצינות?

**תשובה:** לא. בשלוש הרצות יש בדרך כלל 4–9 המלצות, כך שחבילה בדויה אחת מזיזה את השיעור ב-10–25 נקודות אחוז — הרבה יותר מ-5.2% עצמו. כדי להשוות ברצינות צריך מאות המלצות (הרבה הרצות ופרומפטים), ורווח סמך סביב השיעור שנמדד (מפגש 3, חלק י').

**(ג)** למה `check_pypi` מציג את **תאריך ההעלאה הראשונה**? מה המשמעות של חבילה שקיימת, אבל הועלתה לראשונה לפני שבועיים, ושמה זהה לשם שמודלים נוטים להמציא?

**תשובה:** כי קיום ב-PyPI אינו בטיחות. חבילה שהועלתה לראשונה לפני שבועיים, ששמה זהה לשם שמודלים נוטים להמציא, היא סימן אפשרי ל-**slopsquatting**: תוקף רשם את השם הבדוי וממתין שמישהו יתקין אותו בעקבות המלצת AI. חבילה ותיקה (כמו `pandas` מ-2009) היא סימן מרגיע, לא הוכחה.

</div>

<div dir="rtl">

---

# חלק ו' — מסקנה (2 דק')

חמישה משפטים, אחד לכל חלק. כל משפט בנוי כך: **מה מימשתם ומדדתם ← איזו חוליה במנגנון זה מראה ← מה עושים בעקבות זה בעבודה עם AI.**

1. **טוקניזציה:** מימשנו BPE וראינו שרק רצפים שכיחים נמזגים לטוקן אחד ← אוצר המילים משקף שכיחות בנתונים ← עברית ושמות נדירים "עולים" יותר טוקנים, ולכן כותבים קוד ושמות באנגלית ומודדים טוקנים במקום לנחש.
2. **Embeddings:** מימשנו דמיון קוסינוס וראינו שהוא משווה כיוון בלבד ← קרבה במרחב = משמעות דומה ← שם "כמעט נכון" (כמו `get_value`) נראה למודל סביר, ולכן בודקים בתיעוד דווקא שמות שנשמעים מוכרים.
3. **טמפרטורה:** מימשנו softmax יציב ומדדנו שב-$T=1$ רק כ-1% מהתשובות זהות לתשובה המובילה ← דגימה מצטברת על פני מאות טוקנים ← תשובות שונות בין הרצות הן צפויות, ותשובות שונות מאוד הן סימן להגביר בדיקה.
4. **הזיות:** מימשנו בדיקת קיום וגילינו שהיא תופסת שמות אבל לא פרמטרים שנבלעים ב-`**kwargs` ← המודל מייצר שמות סבירים בלי שלב אימות ← אימות סטטי לא מספיק; חובה להריץ ולקרוא תיעוד.
5. **ניסוי החבילות:** מדדנו עקביות (ג'קארד) וקיום ב-PyPI של המלצות אמיתיות ← התפלגות שטוחה בבחירת ספריות + אין אימות פנימי ← בודקים כל חבילה (כולל תאריך העלאה) לפני התקנה, ולא מסיקים משלוש הרצות.

---

# בונוס

**פתרון בונוס 1:** אחרי 20 מיזוגים: `ממוצע → ['ממוצע']`, `ממוצעים → ['ממוצע', 'ים']`, `מספרים → ['מספרים']`. ברגע שיש מספיק עברית בקורפוס, המילים השכיחות הופכות לטוקן אחד — ואפילו סיומת הרבים `ים` נלמדת כיחידה. הבעיה של עברית היא כמות הנתונים, לא השפה עצמה. **פתרון בונוס 2:** שאלה אמפירית — התשובה תלויה בנתונים שנמדדו. ההשערה הסבירה: שני כלים שונים יסכימו **פחות** משלוש הרצות של אותו כלי, כי לכל כלי מודל, נתוני אימון והנחיית מערכת משלו — כלומר התפלגות אחרת. תשובה טובה משווה את ג'קארד בין הכלים לג'קארד בתוך כל כלי, ומסייגת (מעט הרצות).

**1. BPE על עברית.** הוסיפו לקורפוס של חלק א' את המילים `{"ממוצע": 8, "ממוצעים": 3, "מספר": 6, "מספרים": 4}`, הריצו **20** מיזוגים, ופרקו את `ממוצע`, `ממוצעים`, `מספרים`. מה קרה עכשיו? מה זה אומר על הקשר בין כמות העברית בנתוני האימון לבין איכות הטוקניזציה?

**2. ניסוי בין כלים.** חזרו על חלק ה' בכלי AI שני, וחשבו ג'קארד **בין הכלים** (איחוד ההמלצות של כל כלי). האם שני כלים מסכימים יותר או פחות משלוש הרצות של אותו כלי? הסבירו.

</div>

In [23]:
# בונוס 1 — BPE עם עברית בקורפוס
words_he = {"total": 6, "total_price": 4, "total_tax": 3, "price": 5, "tax": 3, "vat": 1, "with": 1,
            "ממוצע": 8, "ממוצעים": 3, "מספר": 6, "מספרים": 4}
corpus_he = [(list(w), f) for w, f in words_he.items()]
merges_he = []
for step in range(20):
    pairs = count_pairs(corpus_he)
    best = max(pairs, key=lambda p: (pairs[p], p))
    merges_he.append(best)
    corpus_he = [(merge_pair(t, best), f) for t, f in corpus_he]

for w in ["ממוצע", "ממוצעים", "מספרים", "total_price", "price_with_vat"]:
    print(f"{w:16} → {tokenize(w, merges_he)}")

ממוצע            → ['ממוצע']
ממוצעים          → ['ממוצע', 'ים']
מספרים           → ['מספרים']
total_price      → ['total_price']
price_with_vat   → ['price', '_', 'w', 'i', 't', 'h', '_', 'v', 'a', 't']


In [24]:
# בונוס 2 — דוגמה (הרשימות להמחשה; יש להחליף בתוצאות אמיתיות משני כלים)
tool_1 = set(run1) | set(run2) | set(run3)
tool_2 = {"pandas", "openpyxl"}
print(f"ג'קארד בין הכלים: {jaccard(tool_1, tool_2):.2f}")

ג'קארד בין הכלים: 0.50


<div dir="rtl">

---

# בדיקה מסכמת — הריצו לפני ההגשה

התא הבא מריץ שוב את כל תאי הבדיקה. לפני ההגשה: **Runtime → Restart session and run all**, וודאו שכל השורות כאן מסומנות `✓`.

</div>

In [25]:
for name, check in [("א' — count_pairs, merge_pair", check_a),
                    ("ב' — cosine", check_b),
                    ("ג' — softmax, p_greedy_path", check_c),
                    ("ד' — audit", check_d),
                    ("ה' — jaccard", check_e)]:
    try:
        check()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"✗ {name}  ← עדיין לא מומש")
    except AssertionError as e:
        print(f"✗ {name}  ← {e}")

✓ א' — count_pairs, merge_pair
✓ ב' — cosine
✓ ג' — softmax, p_greedy_path
✓ ד' — audit
✓ ה' — jaccard
